# Lab 2.4 — Query Transactions with ES|QL

**Before you start:** select **Cell > Run All** to initialize the harness.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong
from elasticsearch import Elasticsearch

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=60)
EMBED_ID = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)
print(f'Harness ready. FAST={FAST}, STRONG={STRONG}')

In [ ]:
# ── YOUR WORK ── Write 3 ES|QL queries ──────────────────────────────────────
# Load dev queries for reference
import json, pathlib
dev = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-esql-queries.json').read_text())
print(f'Dev queries loaded: {len(dev)}')

# Query 1 — Filter (parameterized)
filter_query = (
    "FROM cortex-transactions "
    "| WHERE amount > ?threshold AND origin_country == ?country "
    "| WHERE DATE_DIFF('day', transaction_date, NOW()) <= ?days "
    "| LIMIT 100"
)

# Query 2 — Aggregation (flagged transactions by risk tier)
aggregation_query = (
    "FROM cortex-transactions "
    "| WHERE is_flagged == true "
    "| STATS count = COUNT(*), total = SUM(amount) BY risk_tier "
    "| SORT count DESC"
)

# Query 3 — Weekly bucket
weekly_bucket_query = (
    "FROM cortex-transactions "
    "| WHERE is_flagged == true "
    "| EVAL week = DATE_TRUNC(1 week, transaction_date) "
    "| STATS volume = SUM(amount) BY week, risk_tier "
    "| SORT week DESC, risk_tier"
)

# Test filter query with dev parameters
r = es.esql.query(query=filter_query, params=[{'threshold': 10000}, {'country': 'US'}, {'days': 30}])
print(f'Filter: {len(r["values"])} rows')

In [ ]:
# Save your queries
import json, pathlib
queries = {
    'filter': filter_query,
    'aggregation': aggregation_query,
    'weekly_bucket': weekly_bucket_query,
}
pathlib.Path('/home/elastic/esql-queries.json').write_text(json.dumps(queries, indent=2))
print('Queries saved. Select Check in the sidebar.')